In [1]:
import torch
import torch.nn.functional as F
from unsloth import FastLanguageModel
from transformers import AutoModelForCausalLM, AutoTokenizer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### Dataset Generation

In [2]:
from datasets import load_dataset

TRAIN_SIZE = 2000
dataset = load_dataset("allenai/tulu-3-sft-mixture", split="train", streaming=True)
dataset.shuffle(seed=42)
train_dataset = []
for i, x in enumerate(dataset):
    if i >= TRAIN_SIZE:
        break
    train_dataset.append(dict(messages=x['messages']))

train_dataset[0]

{'messages': [{'content': 'Create a snippet of Terraform HCL code that create an AWS autoscaling group, and an ALB in front to expose an application to internet.',
   'role': 'user'},
  {'content': 'Sure, here\'s an example Terraform HCL code that creates an AWS Autoscaling Group and an Application Load Balancer to expose an application to the internet:\n``` \n# Configure the AWS provider\nprovider "aws" {\n  region = "us-east-1"\n}\n\n# Create a security group to allow traffic to the ALB\nresource "aws_security_group" "alb_sg" {\n  name_prefix = "alb_sg"\n  ingress {\n    from_port = 80\n    to_port = 80\n    protocol = "tcp"\n    cidr_blocks = ["0.0.0.0/0"]\n  }\n}\n\n# Create an ALB and target group\nresource "aws_lb" "alb" {\n  name               = "example-alb"\n  internal           = false\n  load_balancer_type = "application"\n\n  subnets = ["subnet-12345678", "subnet-87654321"]\n\n  security_groups = [aws_security_group.alb_sg.id]\n\n  tags = {\n    Environment = "production"\n

In [3]:
# ----------------------------
# 1) Load student with Unsloth
# ----------------------------
import os

max_seq_length = 1024
STUDENT_MODEL_NAME = "./knowledge-ingestion-test/model/checkpoint-178"
os.path.exists(STUDENT_MODEL_NAME)


True

In [4]:
# ----------------------------
# 1) Load student with Unsloth
# ----------------------------
max_seq_length = 2048
lora_rank = 32

student, tok = FastLanguageModel.from_pretrained(
    model_name=STUDENT_MODEL_NAME,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False,              # training model
)

==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 8. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.4 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [5]:
student.train()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560, padding_idx=151669)
        (layers): ModuleList(
          (0): Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lo

In [7]:
device = next(student.parameters()).device
device

device(type='cuda', index=0)

In [8]:
# ----------------------------
# 2) Load frozen teacher
# ----------------------------
# For THIS minimal implementation, assume teacher/student share tokenizer ids
teacher_name = "Qwen/Qwen3-4B-Instruct-2507"  # example
teacher = AutoModelForCausalLM.from_pretrained(
    teacher_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
teacher.eval()

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [9]:
teacher_tok = AutoTokenizer.from_pretrained(teacher_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [10]:
optimizer = torch.optim.AdamW(student.parameters(), lr=1e-5)

In [11]:
def selected_token_logprobs(model, input_ids, attention_mask=None):
    """
    Returns log p(x_t | x_<t>) for each realized token x_t.
    Shape: [B, T-1]
    """
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits[:, :-1, :]                       # predict token 1..T-1
    targets = input_ids[:, 1:]                          # realized tokens
    logprobs = F.log_softmax(logits, dim=-1)
    token_logprobs = logprobs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    return token_logprobs

In [12]:
@torch.no_grad()
def sample_from_student(prompt, max_new_tokens=256, temperature=1.0, top_p=1.0):
    inputs = tok(prompt, return_tensors="pt").to(device)
    prompt_len = inputs["input_ids"].shape[1]

    sampled = student.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tok.eos_token_id,
        eos_token_id=tok.eos_token_id,
    )
    return sampled, prompt_len

In [13]:
def opd_step(prompt):
    with torch.no_grad():
        full_ids, prompt_len = sample_from_student(prompt)

    full_ids = full_ids.to(device)
    attn = torch.ones_like(full_ids, device=device)

    # 2) recompute student logprobs WITH grad on realized tokens
    student_lp_all = selected_token_logprobs(student, full_ids, attn)

    # token positions corresponding only to generated continuation
    # first generated token is at index prompt_len in full_ids,
    # so its logprob lives at position prompt_len-1 in shifted logprobs
    student_lp = student_lp_all[:, prompt_len - 1 :]

    # 3) teacher logprobs on SAME realized student tokens (no grad)
    with torch.no_grad():
        teacher_ids = full_ids.to(teacher.device)
        teacher_attn = torch.ones_like(teacher_ids, device=teacher.device)
        teacher_lp_all = selected_token_logprobs(teacher, teacher_ids, teacher_attn)
        teacher_lp = teacher_lp_all[:, prompt_len - 1 :].to(student_lp.device)

    # 4) reverse-KL advantage on sampled tokens
    # reverse_kl_t = log pi_student - log pi_teacher
    # advantage_t  = - reverse_kl_t
    advantage = teacher_lp - student_lp.detach()

    # 5) REINFORCE-style surrogate
    loss = -(advantage * student_lp).mean()

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        mean_reverse_kl = (student_lp - teacher_lp).mean().item()

    return {
        "loss": float(loss.item()),
        "mean_reverse_kl": mean_reverse_kl,
        "sample_text": tok.decode(full_ids[0], skip_special_tokens=True),
    }

In [14]:
# ----------------------------
# 3) Train
# ----------------------------
prompts = [
    "Solve: If x^2 - 5x + 6 = 0, what are x?",
    "What is the derivative of x^3 + 2x?",
]

for step in range(10):
    prompt = prompts[step % len(prompts)]
    stats = opd_step(prompt)
    if step % 10 == 0:
        print(step, stats["loss"], stats["mean_reverse_kl"])

--- Logging error ---
Traceback (most recent call last):
  File "/home/lab/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/home/lab/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/home/lab/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/home/lab/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", lin

RuntimeError: Inference tensors cannot be saved for backward. Please do not use Tensors created in inference mode in computation tracked by autograd. To work around this, you can make a clone to get a normal tensor and use it in autograd, or use `torch.no_grad()` instead of `torch.inference_mode()`.